In [1]:
import torch

import triton
import triton.language as tl

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
@triton.jit
def _attn_fwd_inner(
    O_block,
    l_i,
    m_i,
    Q_block,
    K_block_ptr,
    V_block_ptr,
    block_index_q,
    softmax_scale,
    BLOCK_SIZE_Q: tl.constexpr,
    BLOCK_SIZE_KV: tl.constexpr,
    STAGE: tl.constexpr,
    offs_q: tl.constexpr,
    offs_kv: tl.constexpr,
    SEQ_LEN: tl.constexpr,
    ):
  #range of values handled by this stage
  if STAGE == 1:
    #from 0 to the left of the diagonal
    lo, hi = 0, block_index_q * BLOCK_SIZE_Q
  elif STAGE == 2:
    #used only for the block in which there is transition between non-masked and masked keys
    #so essentially the ones of the diagonal
    lo, hi = block_index_q * BLOCK_SIZE_Q, (block_index_q + 1) * BLOCK_SIZE_Q
    lo = tl.multiple_of(lo, BLOCK_SIZE_Q)
  else:
    #only used for non causal attention
    lo, hi = 0, SEQ_LEN

  K_block_ptr = tl.advance(K_block_ptr, (0, lo))
  V_block_ptr = tl.advance(V_block_ptr, (lo, 0))

  #loop over k, v and update the accumulator
  for start_kv in range(lo, hi, BLOCK_SIZE_KV):
    #just lets the compiler know that start_n is a multiple of BLOCK_N, so the compiler can do optimizations
    start_kv = tl.multiple_of(start_kv, BLOCK_SIZE_KV)

    #--computing qk---
    K_block = tl.load(K_block_ptr)
    QK_block = tl.dot(Q_block, K_block)

    if STAGE==2:
      mask = offs_q[:, None] >= (start_kv + offs_kv[None, :])
      OK_block = QK_Block * softmax_scale + tl.where(mask, 0, -1.0e6)
      m_ij = tl.maximum(m_i, tl.max(QK_block, 1))
      QK_block -= m_ij[:, None]
    else:
      #compute the maximum value of qk or keep the old max value
      m_ij = tl.maximum(m_i, tl.max(QK_block, 1)*softmax_scale)
      QK_block = QK_block * softmax_scale - m_ij[:,None]

    #compute the exponential of each dot product, so now we are computing exp(qk_ij - m_ij)
    P_block = tl.math.exp(QK_block)

    #compute the sum by rows of the attention scores
    l_ij = tl.sum(P_block, 1)

    #this is the correction factor for the previous l_i
    alpha = tl.math.exp(m_i - m_ij)

    #apply the correction factor to the previous l_i and add the new l_ij
    l_i = l_i*alpha + l_ij

    V_block = tl.load(V_block_ptr)

    P_block = P_block.to(tl.float16)

    #this computes O_new = P x V + O_old * alpha
    O_block = O_block * alpha[:, None]
    O_block = tl.dot(P_block, V_block, O_block) #same as O += P @ V

    m_i = m_ij

    #move to the next block of K and V
    V_block_ptr = tl.advance(V_block_ptr, (BLOCK_SIZE_KV, 0))
    K_block_ptr = tl.advance(K_block_ptr, (0, BLOCK_SIZE_KV))




return O_block, l_i, m_i















In [ ]:
@triton.jit
def _attn_fwd(
    Q, #BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM , #Q[index_batch, index_head, :, :]
    K, #BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM
    V, #BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM
    softmax_scale,
    M, #BATCH_SIZE, NUM_HEADS, SEQ_LEN
    O, #BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM
    stride_Q_batch,
    stride_Q_head,
    stride_Q_seq,
    stride_Q_dim,
    stride_K_batch,
    stride_K_head,
    stride_K_seq,
    stride_K_dim,
    stride_V_batch,
    stride_V_head,
    stride_V_seq,
    stride_V_dim,
    stride_O_batch,
    stride_O_head,
    stride_O_seq,
    stride_O_dim,
    BATCH_SIZE,
    NUM_HEADS: tl.constexpr,
    SEQ_LEN: tl.constexpr,
    HEAD_DIM: tl.constexpr,
    BLOCK_SIZE_Q: tl.constexpr,
    BLOCK_SIZE_KV: tl.constexpr,
    STAGE: tl.constexpr,

):

  tl.static_assert(BLOCK_SIZE_KV<=HEAD_DIM)

  #this indicates which block in the sequence length to process
  block_index_q = tl.program_id(0)

  #this indicates which head and batch to process, each program is associated with a single head of a single batch
  index_batch_head = tl.progrm_id(1)

  #this indicates which batch this program is assoc. with (each batch has NUM_HEADS heads)
  index_batch = index_batch_head // NUM_HEADS

  #this indicates the position of the head in the batch
  index_head = index_batch_head % NUM_HEADS


  #this allows us to get the (SEQ_LEN, HEAD_DIM) block in the Q, K, V by selecting indexing it by batch and head
  qvk_offset = (
      index_batch.to(tl.int64) * stride_Q_batch
      + index_head.to(tl.int64) * stride_Q_head
  )

  Q_block_ptr = tl.make_block_ptr(
      #Q[index_batch, index_head, block_index_q*BLOCK_SIZE_Q, :]
      base=Q + qvk_offset,
      shape=(SEQ_LEN, HEAD_DIM),
      strides=(stride_Q_seq, stride_Q_dim),
      offsets=(block_index_q*BLOCK_SIZE_Q,0),
      block_shape=(BLOCK_SIZE_Q, HEAD_DIM),
      order=(1,0),
  )

  V_block_ptr = tl.make_block_ptr(
      #V[index_batch, index_head, :, :]
      base=V + qvk_offset,
      shape=(SEQ_LEN, HEAD_DIM),
      strides=(stride_V_seq, stride_V_dim),
      offsets=(0,0),
      block_shape=(BLOCK_SIZE_KV, HEAD_DIM),
      order=(1,0),

  )

  K_block_ptr = tl.make_block_ptr(
      #K[index_batch, index_head, :, :]
      base=K + qvk_offset,
      shape=(HEAD_DIM, SEQ_LEN),
      strides=(stride_K_dim,
              stride_K_seq,
              #we invert the strides wrt Q, so we transpose the matrix
              ),
      offsets=(0,0),
      block_shape=(HEAD_DIM, BLOCK_SIZE_KV),
      order=(0,1),
  )

  O_block_ptr = tl.make_block_ptr(
      #O[index_batch, index_head, :, :]
      base=O + qvk_offset,
      shape = (SEQ_LEN, HEAD_DIM),
      strides=(stride_O_seq, stride_O_dim),
      offsets=(block_index_q*BLOCK_SIZE_Q, 0),
      block_shape=(BLOCK_SIZE_Q, HEAD_DIM),
      order=(1,0),
  )

  #offs_q: the offsets for the tokens in the Q to process
  offs_q = block_index_q * BLOCK_SIZE_Q + tl.arange(0, BLOCK_SIZE_Q)

  #offs_kv: the offsets for the tokens in the K and V sequence to process
  offs_kv = tl.arange(0, BLOCK_SIZE_KV) #we don't skip anything because each row in Q is multiplied to all rows in K and V

  #m_i: the running maximum, we have one for each query
  m_i = tl.zeros([BLOCK_SIZE_Q], dtype=tl.float32) - float("inf")

  #l_i: the running sum/normalization factor, we have one for each query (as we sum the attention scores by rows)
  l_i = tl.zeros([BLOCK_SIZE_Q], dtype=tl.float32) + 1.0 #the +1 was in the original code to make the "log" stable as this is later used in the logsumexp

  #acc: the accumulator for the output, which is a group of rows of the 0 matrix
  O_block = tl.zeros([BLOCK_SIZE_Q, HEAD_DIM], dtype=tl.float32)

  #load the blocks of Q: it will stay in the SRAM throughout
  Q_block = tl.load(Q_block_ptr)

  #Stage : 3 if causal, else 1

  if STAGE==1 or STAGE==3:
    #this step runs for non-causal attention or for the blocks to the left of the diagonal in the causal attention
    O_block, l_i, m_i = _attn_fwd_inner(
        O_block,
        l_i,
        m_i,
        Q_block,
        K_block_ptr,
        V_block_ptr,
        block_index_q,
        softmax_scale,
        BLOCK_SIZE_Q,
        BLOCK_SIZE_KV,
        4 - STAGE,
        offs_q,
        offs_kv,
        SEQ_LEN,

    )

    if STAGE==3:
      #this steps runs for the blocks to the right of the diagonal in the causal attention
      O_block, l_i, m_i = _attn_fwd_inner(
          O_block,
          l_i,
          m_i,
          Q_block,
          K_block_ptr,
          V_block_ptr,
          block_index_q,
          softmax_scale,
          BLOCK_SIZE_Q,
          BLOCK_SIZE_KV,
          2,
          offs_q,
          offs_kv,
          SEQ_LEN,
      )































In [3]:
#each time you want to derive a new operation in torch you need to derive a class from torch.autograd.function
#and it needs to have two functions, the forward pass to produce the output and the backward to compute the gradients

class TritonAttention(torch.autograd.Function):

  @staticmethod
  def forward(ctx, Q, K, V, causal, softmax_scale):
    HEAD_DIM_Q, HEAD_DIM_K = Q.shape[-1], K.shape[-1]
    HEAD_DIM_V = V.shape[-1]

    BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM = Q.shape

    assert HEAD_DIM_Q == HEAD_DIM_K and HEAD_DIM_K == HEAD_DIM_V

    #preallocate the output vector
    O = torch.empty_like(Q)

    stage = 3 if causal else 1

    grid = lambda args: (
        #ceil(SEQ_LEN/BLOCK_SIZE_Q) = how many blocks of Q we have
        triton.cdiv(SEQ_LEN, args["BLOCK_SIZE_Q"]), #which group of queries are we going to work with?
        BATCH_SIZE*NUM_HEADS, #which head of which batch element are we going to work with?
        1, #Z in the CUDA launch grid
    )

    #Number of parallel programs: [BATCH_SIZE * NUM_HEADS * NUM_BLOCKS_Q]


    #M is the logsumexp for the backward pass, one for each query
    M = torch.empty(
        (BATCH_SIZE, NUM_HEADS, SEQ_LEN), device=Q.device, dtype=torch.float32
    )

    _attn_fwd[grid](
        Q=Q,
        K=K,
        V=V,
        softmax_scale=softmax_scale,
        M=M,
        O=O,
        stride_Q_batch=Q.stride(0),
        stride_Q_head=Q.stride(1),
        stride_Q_seq=Q.stride(2),
        stride_Q_dim=Q.stride(3),
        stride_K_batch=K.stride(0),
        stride_K_head=K.stride(1),
        stride_K_seq=K.stride(2),
        stride_K_dim=K.stride(3),
        stride_V_batch=V.stride(0),
        stride_V_head=V.stride(1),
        stride_V_seq=V.stride(2),
        stride_V_dim=V.stride(3),
        stride_O_batch=O.stride(0),
        stride_O_head=O.stride(1),
        stride_O_seq=O.stride(2),
        stride_O_dim=O.stride(3),
        BATCH_SIZE=Q.shape[0],
        NUM_HEADS=Q.shape[1],
        SEQ_LEN=Q.shape[2],
        HEAD_DIM=HEAD_DIM_K,
        STAGE=stage,
    )

    ctx.save_for_backward(Q, K, V, O, M)
    ctx.grid = grid
    ctx.softmax_scale = softmax_scale
    ctx.HEAD_DIM = HEAD_DIM_K
    ctx.causal = causal
    return 0





In [ ]:
def test_op(BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM, causal, dtype=torch.float16):
  Q = (
      torch.empty(
          (BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM), dtype=dtype, device="cuda"
      )
      .normal_(mean=0.0, std=0.5)
      .requires_grad_()
  )
  K = (
      torch.empty(
          (BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM), dtype=dtype, device="cuda"
      )
      .normal_(mean=0.0, std=0.5)
      .requires_grad_()
  )
  V = (
      torch.empty(
          (BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM), dtype=dtype, device="cuda"
      )
      .normal_(mean=0.0, std=0.5)
      .requires_grad_()
  )

  softmax_scale = 1/(HEAD_DIM**0.5) #QK^T/sqrt(HEAD_DIM)
  dO = torch.rand_like(Q) #Needed for the backward pass

  #reference implementation
  MASK = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN), device="cuda")
  P = torch.matmul(Q, K.transpose(2,3))*softmax_scale
  if causal:
    P[:, :, MASK==0] = float["-inf"]
  P = torch.softmax(P.float(), dim=-1).half()
  ref_O = torch.matmul(P, V)
  ref_O.backward(dO)
  ref_dV, V.grad = V.grad.clone(), None
  ref_dK, K.grad = K.grad.clone(), None
  ref_dQ, Q.grad = Q.grad.clone(), None

  #triton implementation
  tri_out = TritonAttention.apply(Q, K, V, causal, softmax_scale).half()
  tri_out.backward(dO)
  tri_dV, V.grad = V.grad.clone(), None
  tri_dK, K.grad = K.grad.clone(), None
  tri_dQ, Q.grad = Q.grad.clone(), None

  #compare
  rtol = 0.0
  atol = 1e-2
  assert torch.allclose(ref_O, tri_out, atol=atol, rtol=rtol)
  assert torch.allclose(ref_dK, tri_dK, atol=atol, rtol=rtol)
  assert torch.allclose(ref_dV, tri_dV, atol=atol, rtol=rtol)
  assert torch.allclose(ref_dQ, tri_dQ, atol=atol, rtol=rtol)


